# 04 — Authentication

`edr-xarray` does not bake any authentication scheme into the library.
Instead, every HTTP request flows through one of two extension points:

1. The `session` keyword — accepts a pre-configured `httpx.Client`.
   The client's headers and auth are applied to every request.
2. Subclassing `EdrDataStore._request` — gives full control, including
   per-request token refresh.

This notebook shows both, plus a custom timeout.

In [ ]:
import httpx

server = "https://edr.example.com"  # replace with your EDR server root

collections = httpx.get(f"{server}/collections").raise_for_status().json()["collections"]
collection_id = collections[0]["id"]  # or pick any id from the list
collection_url = f"{server}/collections/{collection_id}"

## Pattern 1: API key via custom header

In [ ]:
%pip install -q edr-xarray

In [ ]:
client = httpx.Client(headers={"X-Api-Key": "demo-secret-key"})
ds = xr.open_dataset(collection_url, engine="edr", session=client)
print("opened with API key; dims:", dict(ds.dims))
ds.close()
client.close()  # injected sessions are NOT closed by edr-xarray

## Pattern 2: Bearer token

In [ ]:
client = httpx.Client(headers={"Authorization": "Bearer demo-jwt-token"})
ds = xr.open_dataset(collection_url, engine="edr", session=client)
print("opened with Bearer token; dims:", dict(ds.dims))
ds.close()
client.close()

## Pattern 3: Basic auth

In [ ]:
client = httpx.Client(auth=("alice", "wonderland"))
ds = xr.open_dataset(collection_url, engine="edr", session=client)
print("opened with HTTP Basic; dims:", dict(ds.dims))
ds.close()
client.close()

## Pattern 4: Custom timeout

The `timeout` keyword is forwarded to the *owned* `httpx.Client`
(i.e. when you do not supply your own `session`). When you inject a
session, set the timeout on the session itself.

In [ ]:
ds = xr.open_dataset(collection_url, engine="edr", timeout=60.0)
print("opened with 60s timeout; dims:", dict(ds.dims))
ds.close()

# When using your own session:
client = httpx.Client(timeout=60.0)
ds2 = xr.open_dataset(collection_url, engine="edr", session=client)
ds2.close()
client.close()

## Pattern 5: Subclass for dynamic auth (token refresh)

For OAuth-style flows where the access token must be refreshed
periodically, override `EdrDataStore._request`. Every cube fetch
flows through this hook, so you can mutate `headers` per call.

In [ ]:
import time
from collections.abc import Mapping

from edr_xarray import EdrDataStore


class TokenRefreshingStore(EdrDataStore):
    """Demonstrate per-request token refresh.

    In production you would probably cache the token until a few
    seconds before its `exp` claim. Here we just stamp the current
    epoch into the header so each request carries a fresh value.
    """

    def _request(self, method, url, *, params=None, headers=None):
        merged: dict[str, str] = dict(headers or {})
        merged["Authorization"] = f"Bearer demo-token-{int(time.time())}"
        return super()._request(method, url, params=params, headers=merged)


store = TokenRefreshingStore(collection_url=collection_url)
ds = store.build_dataset()
print("opened via TokenRefreshingStore; dims:", dict(ds.dims))
_ = ds["temperature"].values  # triggers a fetch with a fresh token
ds.close()